<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **Repositorio profesional, entornos, dependencias y configuración**

## Del notebook aislado al proyecto reproducible

En este laboratorio construiremos físicamente un proyecto Python profesional.

No nos limitaremos a mostrar un árbol de carpetas: el notebook generará y validará de forma ejecutable:

- estructura de directorios;
- paquete `src/`;
- `tests/`;
- `data/raw` y `data/processed`;
- `README.md`;
- `.gitignore`;
- `pyproject.toml`;
- `.env.example`;
- `.env`;
- carga de variables con `python-dotenv`;
- configuración JSON;
- configuración YAML;
- validación de parámetros;
- módulo reusable de configuración;
- módulo de pipeline;
- script de entrada;
- comprobaciones automáticas de integridad.

Seguiremos el patrón del curso:

> **crear → inspeccionar → provocar inconsistencia → detectar → corregir → validar**

## Objetivos de aprendizaje

Al finalizar deberías poder:

1. Diseñar la estructura de un proyecto Python profesional.
2. Crearla automáticamente con `pathlib`.
3. Comprender la función de `src/`, `tests/`, `data/`, `notebooks/`, `scripts/` y `config/`.
4. Diferenciar entorno virtual y proyecto.
5. Crear un entorno virtual con `venv`.
6. Entender activación en Windows y Linux/macOS.
7. Definir dependencias en `pyproject.toml`.
8. Separar dependencias de runtime y desarrollo.
9. Crear un `.gitignore`.
10. Gestionar configuración externa.
11. Utilizar `.env.example`.
12. Crear y cargar un `.env` real.
13. Utilizar `python-dotenv`.
14. Evitar secretos dentro del repositorio.
15. Leer configuración JSON.
16. Leer configuración YAML.
17. Validar configuración.
18. Crear un módulo de configuración reusable.
19. Crear un script de entrada.
20. Verificar que la estructura generada es consistente.

## 1. Preparación del entorno

In [ ]:
from pathlib import Path
import json
import os
import shutil
import sys
import textwrap

import pandas as pd

try:
    import yaml
    YAML_DISPONIBLE = True
except ImportError:
    YAML_DISPONIBLE = False

try:
    from dotenv import load_dotenv
    DOTENV_DISPONIBLE = True
except ImportError:
    DOTENV_DISPONIBLE = False

print("Python:", sys.version)
print("PyYAML disponible:", YAML_DISPONIBLE)
print("python-dotenv disponible:", DOTENV_DISPONIBLE)

## 2. Diseñar la estructura del proyecto

Utilizaremos esta estructura:

```text
proyecto_data/
├── data/
│   ├── raw/
│   └── processed/
├── notebooks/
├── src/
│   └── proyecto_data/
│       ├── __init__.py
│       ├── config.py
│       └── pipeline.py
├── tests/
├── scripts/
├── config/
├── .env
├── .env.example
├── .gitignore
├── pyproject.toml
└── README.md
```

La estructura separa responsabilidades y evita que lógica, configuración, datos y artefactos se mezclen.

## 2.1 Crear estructura automáticamente

In [ ]:
raiz = Path("proyecto_data_demo")

if raiz.exists():
    shutil.rmtree(raiz)

carpetas = [
    raiz / "data" / "raw",
    raiz / "data" / "processed",
    raiz / "notebooks",
    raiz / "src" / "proyecto_data",
    raiz / "tests",
    raiz / "scripts",
    raiz / "config"
]

for carpeta in carpetas:
    carpeta.mkdir(
        parents=True,
        exist_ok=True
    )

print("Estructura creada en:")
print(raiz.resolve())

## 2.2 Crear ficheros mínimos

In [ ]:
ficheros = {
    raiz / "src" / "proyecto_data" / "__init__.py": "",
    raiz / "tests" / "__init__.py": "",
    raiz / "README.md": "# Proyecto Data\n",
    raiz / ".env.example": "API_KEY=\nDATABASE_URL=\nENVIRONMENT=development\n"
}

for ruta, contenido in ficheros.items():
    ruta.write_text(
        contenido,
        encoding="utf-8"
    )

print("Ficheros mínimos creados.")

## 2.3 Inspeccionar árbol

In [ ]:
for ruta in sorted(raiz.rglob("*")):
    nivel = len(
        ruta.relative_to(raiz).parts
    ) - 1

    sangria = "    " * nivel
    simbolo = "📁" if ruta.is_dir() else "📄"

    print(
        f"{sangria}{simbolo} {ruta.name}"
    )

### Interpretación

La estructura ya existe físicamente en disco.

Esto permite que todos los alumnos partan de un proyecto reproducible y automatizable.

## 3. Validar estructura

In [ ]:
rutas_obligatorias = [
    raiz / "data" / "raw",
    raiz / "data" / "processed",
    raiz / "notebooks",
    raiz / "src" / "proyecto_data",
    raiz / "tests",
    raiz / "scripts",
    raiz / "config",
    raiz / "README.md"
]

estado = {
    str(ruta.relative_to(raiz)):
        ruta.exists()
    for ruta in rutas_obligatorias
}

pd.Series(estado, name="existe")

## 3.1 Provocar inconsistencia estructural

In [ ]:
ruta_problematica = (
    raiz
    / "data"
    / "processed"
)

shutil.rmtree(
    ruta_problematica
)

print(
    "¿Existe processed?:",
    ruta_problematica.exists()
)

## 3.2 Detectar la inconsistencia

In [ ]:
faltantes = [
    ruta
    for ruta in rutas_obligatorias
    if not ruta.exists()
]

[
    str(ruta.relative_to(raiz))
    for ruta in faltantes
]

## 3.3 Corregir automáticamente

In [ ]:
for ruta in faltantes:
    if ruta.suffix:
        ruta.parent.mkdir(
            parents=True,
            exist_ok=True
        )
        ruta.touch()
    else:
        ruta.mkdir(
            parents=True,
            exist_ok=True
        )

print(
    "Faltantes después:",
    [
        str(r.relative_to(raiz))
        for r in rutas_obligatorias
        if not r.exists()
    ]
)

### Interpretación

Convertimos una expectativa de estructura en una comprobación ejecutable.

Eso permite detectar errores antes de ejecutar el pipeline.

## 4. Entornos virtuales

Un entorno virtual permite aislar dependencias del proyecto.

Comando recomendado:

```bash
python -m venv .venv
```

### Windows PowerShell

```powershell
.\.venv\Scripts\Activate.ps1
```

### Linux/macOS

```bash
source .venv/bin/activate
```

Desactivar:

```bash
deactivate
```

El entorno virtual **no sustituye** a `pyproject.toml`:

- `.venv` aísla paquetes instalados;
- `pyproject.toml` describe el proyecto y sus dependencias.

## 4.1 Mostrar comandos

In [ ]:
comandos = {
    "crear_entorno":
        "python -m venv .venv",

    "activar_windows":
        ".\\.venv\\Scripts\\Activate.ps1",

    "activar_linux_mac":
        "source .venv/bin/activate",

    "desactivar":
        "deactivate"
}

for nombre, comando in comandos.items():
    print(
        f"{nombre}: {comando}"
    )

## 4.2 Detectar si Python está dentro de un venv

In [ ]:
dentro_venv = (
    sys.prefix
    != sys.base_prefix
)

print(
    "Dentro de entorno virtual:",
    dentro_venv
)

print(
    "sys.prefix:",
    sys.prefix
)

print(
    "sys.base_prefix:",
    sys.base_prefix
)

## 5. Dependencias con `pyproject.toml`

In [ ]:
pyproject = '''
[project]
name = "proyecto-data"
version = "0.1.0"
description = "Proyecto profesional de analítica de datos"
readme = "README.md"
requires-python = ">=3.11"

dependencies = [
    "pandas>=2.2",
    "numpy>=1.26",
    "pyyaml>=6.0",
    "python-dotenv>=1.0"
]

[project.optional-dependencies]
dev = [
    "pytest>=8.0",
    "ruff>=0.6",
    "black>=24.0",
    "mypy>=1.10"
]

[build-system]
requires = ["setuptools>=69"]
build-backend = "setuptools.build_meta"

[tool.setuptools.packages.find]
where = ["src"]

[tool.ruff]
line-length = 100

[tool.black]
line-length = 100
'''

ruta_pyproject = (
    raiz
    / "pyproject.toml"
)

ruta_pyproject.write_text(
    textwrap.dedent(pyproject).strip()
    + "\n",
    encoding="utf-8"
)

print(
    ruta_pyproject.read_text(
        encoding="utf-8"
    )
)

### Interpretación

Distinguimos:

- dependencias de ejecución;
- dependencias opcionales de desarrollo.

`python-dotenv` entra como dependencia porque lo usaremos para cargar configuración desde `.env`.

## 5.1 `pip freeze` vs dependencias declaradas

`pip freeze` captura **todo lo instalado** en el entorno.

`pyproject.toml` debería describir las dependencias **intencionadas** del proyecto.

No debemos confundir:

> lo que casualmente tengo instalado

con

> lo que realmente necesita el proyecto

## 5.2 Instalación editable

In [ ]:
comandos_instalacion = [
    "python -m pip install -e .",
    'python -m pip install -e ".[dev]"'
]

for comando in comandos_instalacion:
    print(comando)

## 6. Crear `.gitignore`

In [ ]:
gitignore = '''
.venv/
__pycache__/
*.pyc
.ipynb_checkpoints/
.env
dist/
build/
*.egg-info/
.pytest_cache/
.mypy_cache/
.ruff_cache/
data/raw/*
data/processed/*
!data/raw/.gitkeep
!data/processed/.gitkeep
'''

ruta_gitignore = (
    raiz
    / ".gitignore"
)

ruta_gitignore.write_text(
    textwrap.dedent(gitignore).strip()
    + "\n",
    encoding="utf-8"
)

print(
    ruta_gitignore.read_text(
        encoding="utf-8"
    )
)

### Interpretación

El `.gitignore` evita versionar:

- entorno virtual;
- cachés;
- secretos;
- artefactos;
- datos pesados.

## 7. `.env.example` y `.env`

Patrón recomendado:

```text
.env.example   → sí se versiona
.env           → no se versiona
```

`.env.example` documenta qué variables necesita el proyecto sin contener secretos reales.

## 7.1 Crear `.env.example`

In [ ]:
env_example = '''
API_KEY=
DATABASE_URL=
ENVIRONMENT=development
'''

ruta_env_example = (
    raiz
    / ".env.example"
)

ruta_env_example.write_text(
    textwrap.dedent(env_example).strip()
    + "\n",
    encoding="utf-8"
)

print(
    ruta_env_example.read_text(
        encoding="utf-8"
    )
)

## 7.2 Crear `.env` de demostración

In [ ]:
env_demo = '''
API_KEY=clave_demo_no_real
DATABASE_URL=sqlite:///demo.db
ENVIRONMENT=development
'''

ruta_env = (
    raiz
    / ".env"
)

ruta_env.write_text(
    textwrap.dedent(env_demo).strip()
    + "\n",
    encoding="utf-8"
)

print(
    "Fichero .env creado:",
    ruta_env.exists()
)

print(
    "No mostramos su contenido completo por buena práctica."
)

### Importante

En un proyecto real:

- `.env` puede contener secretos;
- `.env` no debe subirse a Git;
- `.env.example` sí se comparte.

## 7.3 Cargar `.env` con `python-dotenv`

In [ ]:
if DOTENV_DISPONIBLE:

    load_dotenv(
        dotenv_path=ruta_env,
        override=True
    )

    print(
        "ENVIRONMENT:",
        os.getenv(
            "ENVIRONMENT"
        )
    )

    print(
        "DATABASE_URL configurada:",
        bool(
            os.getenv(
                "DATABASE_URL"
            )
        )
    )

    print(
        "API_KEY configurada:",
        bool(
            os.getenv(
                "API_KEY"
            )
        )
    )

else:
    print(
        "Instalar python-dotenv para ejecutar esta sección."
    )

### Interpretación

`load_dotenv()` carga las variables del fichero `.env` en el entorno del proceso.

Después podemos recuperarlas con `os.getenv()`.

## 7.4 Provocar variable obligatoria ausente

In [ ]:
os.environ.pop(
    "DATABASE_URL",
    None
)

print(
    "DATABASE_URL:",
    os.getenv(
        "DATABASE_URL"
    )
)

## 7.5 Validar variables de entorno

In [ ]:
def validar_variables_entorno(
    requeridas
):
    faltantes = [
        nombre
        for nombre in requeridas
        if not os.getenv(
            nombre
        )
    ]

    if faltantes:
        raise ValueError(
            "Faltan variables de entorno: "
            + ", ".join(
                faltantes
            )
        )


try:
    validar_variables_entorno(
        [
            "API_KEY",
            "DATABASE_URL",
            "ENVIRONMENT"
        ]
    )

except ValueError as error:
    print(
        type(error).__name__,
        "->",
        error
    )

## 7.6 Corregir recargando `.env`

In [ ]:
if DOTENV_DISPONIBLE:

    load_dotenv(
        dotenv_path=ruta_env,
        override=True
    )

    validar_variables_entorno(
        [
            "API_KEY",
            "DATABASE_URL",
            "ENVIRONMENT"
        ]
    )

    print(
        "Variables obligatorias correctas."
    )

## 7.7 Comprobar que `.env` está ignorado

In [ ]:
contenido_gitignore = (
    ruta_gitignore
    .read_text(
        encoding="utf-8"
    )
    .splitlines()
)

print(
    ".env ignorado:",
    ".env" in contenido_gitignore
)

### Interpretación

Ahora sí hemos cubierto el ciclo completo:

```text
.env.example
    ↓
crear .env local
    ↓
load_dotenv()
    ↓
os.getenv()
    ↓
validar
    ↓
comprobar que .env no se versiona
```

## 8. Configuración JSON

In [ ]:
config_json = {
    "seed": 42,
    "input_path":
        "data/raw/ventas.csv",
    "output_path":
        "data/processed/resumen.parquet",
    "threshold":
        500,
    "environment":
        "development"
}

ruta_json = (
    raiz
    / "config"
    / "config.json"
)

ruta_json.write_text(
    json.dumps(
        config_json,
        indent=2
    ),
    encoding="utf-8"
)

print(
    ruta_json.read_text(
        encoding="utf-8"
    )
)

## 8.1 Leer JSON

In [ ]:
config_cargada_json = json.loads(
    ruta_json.read_text(
        encoding="utf-8"
    )
)

config_cargada_json

## 9. Configuración YAML

In [ ]:
if YAML_DISPONIBLE:

    config_yaml = {
        "seed": 42,
        "paths": {
            "input":
                "data/raw/ventas.csv",

            "output":
                "data/processed/resumen.parquet"
        },
        "pipeline": {
            "threshold":
                500,
            "drop_duplicates":
                True
        }
    }

    ruta_yaml = (
        raiz
        / "config"
        / "config.yaml"
    )

    with ruta_yaml.open(
        "w",
        encoding="utf-8"
    ) as fichero:

        yaml.safe_dump(
            config_yaml,
            fichero,
            sort_keys=False
        )

    print(
        ruta_yaml.read_text(
            encoding="utf-8"
        )
    )

else:
    print(
        "Instalar PyYAML para ejecutar esta sección."
    )

## 9.1 Leer YAML

In [ ]:
if YAML_DISPONIBLE:

    with ruta_yaml.open(
        "r",
        encoding="utf-8"
    ) as fichero:

        config_cargada_yaml = (
            yaml.safe_load(
                fichero
            )
        )

    config_cargada_yaml

## 10. `.env` vs JSON vs YAML

In [ ]:
comparativa_config = pd.DataFrame({
    "formato": [
        ".env",
        "JSON",
        "YAML"
    ],
    "uso_principal": [
        "Variables de entorno y secretos locales",
        "Configuración estructurada simple",
        "Configuración estructurada legible"
    ],
    "admite_jerarquia": [
        False,
        True,
        True
    ],
    "versionar_con_secretos": [
        False,
        False,
        False
    ]
})

comparativa_config

### Interpretación

No son equivalentes.

- `.env`: muy adecuado para variables sensibles y valores dependientes del entorno.
- JSON/YAML: adecuados para configuración estructurada.
- Los secretos no deben escribirse en JSON/YAML versionados.

## 11. Provocar configuración inválida

In [ ]:
config_invalida = {
    "seed": "cuarenta_y_dos",
    "threshold": -100,
    "input_path": ""
}

config_invalida

## 11.1 Validar configuración

In [ ]:
def validar_configuracion(
    config
):
    errores = []

    if not isinstance(
        config.get(
            "seed"
        ),
        int
    ):
        errores.append(
            "seed debe ser int"
        )

    if (
        "threshold"
        in config
        and config[
            "threshold"
        ] < 0
    ):
        errores.append(
            "threshold no puede ser negativo"
        )

    if (
        "input_path"
        in config
        and not config[
            "input_path"
        ]
    ):
        errores.append(
            "input_path no puede estar vacío"
        )

    if errores:
        raise ValueError(
            "Configuración inválida: "
            + "; ".join(
                errores
            )
        )


try:
    validar_configuracion(
        config_invalida
    )

except ValueError as error:
    print(
        type(error).__name__,
        "->",
        error
    )

## 11.2 Corregir configuración

In [ ]:
config_valida = {
    "seed": 42,
    "threshold": 500,
    "input_path":
        "data/raw/ventas.csv"
}

validar_configuracion(
    config_valida
)

print(
    "Configuración válida."
)

## 12. Crear módulo reusable `config.py`

In [ ]:
config_py = '''
from pathlib import Path
import json
import os

try:
    import yaml
except ImportError:
    yaml = None

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None


def cargar_env(ruta_env):
    if load_dotenv is None:
        raise ImportError(
            "python-dotenv no está instalado."
        )

    load_dotenv(
        dotenv_path=ruta_env,
        override=True
    )


def obtener_variable_entorno(nombre):
    valor = os.getenv(nombre)

    if not valor:
        raise ValueError(
            f"Falta variable obligatoria: {nombre}"
        )

    return valor


def cargar_configuracion(ruta):
    ruta = Path(ruta)

    if not ruta.exists():
        raise FileNotFoundError(
            f"No existe configuración: {ruta}"
        )

    if ruta.suffix == ".json":
        return json.loads(
            ruta.read_text(
                encoding="utf-8"
            )
        )

    if ruta.suffix in {".yaml", ".yml"}:
        if yaml is None:
            raise ImportError(
                "PyYAML no está instalado."
            )

        with ruta.open(
            "r",
            encoding="utf-8"
        ) as fichero:
            return yaml.safe_load(
                fichero
            )

    raise ValueError(
        f"Formato no soportado: {ruta.suffix}"
    )
'''

ruta_config_py = (
    raiz
    / "src"
    / "proyecto_data"
    / "config.py"
)

ruta_config_py.write_text(
    textwrap.dedent(
        config_py
    ),
    encoding="utf-8"
)

print(
    ruta_config_py.read_text(
        encoding="utf-8"
    )
)

### Interpretación

El módulo centraliza:

- `.env`;
- JSON;
- YAML;
- variables obligatorias.

La lógica del pipeline no necesita conocer cómo se carga cada formato.

## 13. Crear módulo `pipeline.py`

In [ ]:
pipeline_py = '''
import pandas as pd

def preparar_ventas(
    datos: pd.DataFrame,
    threshold: float = 500
) -> pd.DataFrame:

    salida = datos.copy()

    salida["importe"] = (
        salida["unidades"]
        * salida["precio"]
    )

    return salida.loc[
        salida["importe"]
        >= threshold
    ].copy()


def resumir_ventas(
    datos: pd.DataFrame
) -> pd.DataFrame:

    return (
        datos.groupby(
            "canal",
            as_index=False
        )
        .agg(
            facturacion=("importe", "sum"),
            operaciones=("importe", "size")
        )
    )
'''

ruta_pipeline_py = (
    raiz
    / "src"
    / "proyecto_data"
    / "pipeline.py"
)

ruta_pipeline_py.write_text(
    textwrap.dedent(
        pipeline_py
    ),
    encoding="utf-8"
)

print(
    ruta_pipeline_py.read_text(
        encoding="utf-8"
    )
)

## 14. Crear script de entrada

In [ ]:
main_py = '''
from pathlib import Path
import sys

import pandas as pd

ROOT = (
    Path(__file__)
    .resolve()
    .parents[1]
)

SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(
        0,
        str(SRC)
    )

from proyecto_data.config import (
    cargar_configuracion,
    cargar_env,
    obtener_variable_entorno
)

from proyecto_data.pipeline import (
    preparar_ventas,
    resumir_ventas
)


def main():
    cargar_env(
        ROOT / ".env"
    )

    entorno = obtener_variable_entorno(
        "ENVIRONMENT"
    )

    config = cargar_configuracion(
        ROOT
        / "config"
        / "config.json"
    )

    datos = pd.DataFrame({
        "canal": [
            "web",
            "web",
            "tienda"
        ],
        "unidades": [
            2,
            5,
            3
        ],
        "precio": [
            300,
            120,
            200
        ]
    })

    preparados = preparar_ventas(
        datos,
        threshold=config[
            "threshold"
        ]
    )

    resumen = resumir_ventas(
        preparados
    )

    print(
        "Entorno:",
        entorno
    )

    print(
        resumen
    )


if __name__ == "__main__":
    main()
'''

ruta_main = (
    raiz
    / "scripts"
    / "main.py"
)

ruta_main.write_text(
    textwrap.dedent(
        main_py
    ),
    encoding="utf-8"
)

print(
    ruta_main.read_text(
        encoding="utf-8"
    )
)

## 15. Crear README reproducible

In [ ]:
readme = '''
# Proyecto Data

## Crear entorno

```bash
python -m venv .venv
```

### Windows

```powershell
.\\.venv\\Scripts\\Activate.ps1
```

### Linux/macOS

```bash
source .venv/bin/activate
```

## Instalar dependencias

```bash
python -m pip install -e .
```

Para desarrollo:

```bash
python -m pip install -e ".[dev]"
```

## Configuración

Copiar:

```text
.env.example
```

a:

```text
.env
```

y rellenar las variables requeridas.

## Ejecutar

```bash
python scripts/main.py
```

## Estructura

- `data/raw`: datos originales.
- `data/processed`: datos derivados.
- `notebooks`: análisis exploratorio.
- `src/proyecto_data`: código reusable.
- `tests`: pruebas.
- `config`: configuración JSON/YAML.
- `.env`: variables locales y secretos.
- `scripts`: puntos de entrada.
'''

ruta_readme = (
    raiz
    / "README.md"
)

ruta_readme.write_text(
    textwrap.dedent(
        readme
    ).strip()
    + "\n",
    encoding="utf-8"
)

print(
    ruta_readme.read_text(
        encoding="utf-8"
    )
)

## 16. Validar integridad del proyecto

In [ ]:
ficheros_clave = [
    raiz / "pyproject.toml",
    raiz / ".gitignore",
    raiz / ".env.example",
    raiz / ".env",
    raiz / "README.md",
    raiz / "config" / "config.json",
    raiz / "src" / "proyecto_data" / "__init__.py",
    raiz / "src" / "proyecto_data" / "config.py",
    raiz / "src" / "proyecto_data" / "pipeline.py",
    raiz / "scripts" / "main.py"
]

tabla_estado = pd.DataFrame([
    {
        "ruta":
            str(
                ruta.relative_to(
                    raiz
                )
            ),
        "existe":
            ruta.exists(),
        "bytes":
            (
                ruta.stat().st_size
                if ruta.exists()
                else 0
            )
    }
    for ruta in ficheros_clave
])

tabla_estado

## 16.1 Provocar fichero vacío

In [ ]:
contenido_original_pipeline = (
    ruta_pipeline_py
    .read_text(
        encoding="utf-8"
    )
)

ruta_pipeline_py.write_text(
    "",
    encoding="utf-8"
)

print(
    "Bytes pipeline.py:",
    ruta_pipeline_py.stat().st_size
)

## 16.2 Detectar integridad

In [ ]:
def validar_proyecto(
    raiz
):
    errores = []

    obligatorios = [
        "pyproject.toml",
        "README.md",
        ".gitignore",
        ".env.example",
        "src/proyecto_data/__init__.py",
        "src/proyecto_data/config.py",
        "src/proyecto_data/pipeline.py",
        "scripts/main.py"
    ]

    for relativo in obligatorios:
        ruta = (
            raiz
            / relativo
        )

        if not ruta.exists():
            errores.append(
                f"Falta: {relativo}"
            )

        elif (
            ruta.is_file()
            and ruta.stat().st_size == 0
            and ruta.name
            != "__init__.py"
        ):
            errores.append(
                f"Fichero vacío: {relativo}"
            )

    return errores


errores_proyecto = (
    validar_proyecto(
        raiz
    )
)

errores_proyecto

## 16.3 Corregir y volver a validar

In [ ]:
ruta_pipeline_py.write_text(
    contenido_original_pipeline,
    encoding="utf-8"
)

print(
    "Errores restantes:",
    validar_proyecto(
        raiz
    )
)

## 17. Configuración por entorno

In [ ]:
config_por_entorno = {
    "development": {
        "debug": True,
        "threshold": 100
    },
    "production": {
        "debug": False,
        "threshold": 500
    }
}

entorno_actual = os.getenv(
    "ENVIRONMENT",
    "production"
)

config_activa = (
    config_por_entorno[
        entorno_actual
    ]
)

config_activa

### Interpretación

Una misma base de código puede comportarse de forma distinta según el entorno sin modificar el código fuente.

## 18. Mala práctica: secreto hardcodeado

In [ ]:
codigo_malo = '''
API_KEY = "mi_clave_secreta_real"
'''

print(
    codigo_malo
)

### Problema

Un secreto hardcodeado puede terminar en:

- Git;
- logs;
- forks;
- copias;
- historial del repositorio.

Aunque se elimine posteriormente, puede permanecer en el historial.

## 18.1 Alternativa correcta

In [ ]:
api_key = os.getenv(
    "API_KEY"
)

if api_key:
    print(
        "API_KEY disponible."
    )
else:
    print(
        "API_KEY no configurada."
    )

No imprimimos el valor real.

Solo comprobamos si existe.

## 19. Crear `.gitkeep`

In [ ]:
for carpeta in [
    raiz / "data" / "raw",
    raiz / "data" / "processed"
]:
    (
        carpeta
        / ".gitkeep"
    ).touch(
        exist_ok=True
    )

print(
    ".gitkeep creados."
)

Git no versiona carpetas vacías.

`.gitkeep` es una convención habitual para conservarlas en la estructura.

## 20. Visualizar estructura final

In [ ]:
for ruta in sorted(
    raiz.rglob("*")
):
    nivel = (
        len(
            ruta.relative_to(
                raiz
            ).parts
        )
        - 1
    )

    sangria = (
        "    " * nivel
    )

    simbolo = (
        "📁"
        if ruta.is_dir()
        else "📄"
    )

    print(
        f"{sangria}{simbolo} {ruta.name}"
    )

## 21. Mini-práctica integrada

Construye automáticamente un proyecto para un caso de churn.

Debe contener:

```text
churn_project/
├── data/raw
├── data/processed
├── notebooks
├── src/churn_project
├── tests
├── scripts
├── config
├── .env
├── .env.example
├── .gitignore
├── pyproject.toml
└── README.md
```

Además debe incluir:

- configuración JSON;
- configuración YAML;
- variables con `.env`;
- `python-dotenv`;
- módulo de configuración;
- módulo de features;
- script `main.py`;
- dependencias runtime/dev;
- validación automática de integridad.

## 22. Checklist de proyecto reproducible

- [ ] ¿Existe una raíz clara del proyecto?
- [ ] ¿El código reusable está en `src/`?
- [ ] ¿Los tests están separados?
- [ ] ¿Los datos raw y processed están separados?
- [ ] ¿Existe `README.md`?
- [ ] ¿Existe `.gitignore`?
- [ ] ¿`.env` está ignorado?
- [ ] ¿Existe `.env.example`?
- [ ] ¿Las dependencias están declaradas?
- [ ] ¿Se distinguen runtime y dev?
- [ ] ¿Se carga `.env` con `python-dotenv`?
- [ ] ¿Se validan variables obligatorias?
- [ ] ¿JSON/YAML contienen configuración no sensible?
- [ ] ¿Los secretos están fuera del repositorio?
- [ ] ¿Existe un script de entrada?
- [ ] ¿Se puede comprobar automáticamente la estructura?

## 23. Ejercicios finales

### Ejercicio 1 — Estructura
Crea un proyecto completo automáticamente con `pathlib`.

### Ejercicio 2 — Validación
Comprueba programáticamente que todas las rutas obligatorias existen.

### Ejercicio 3 — Inconsistencia
Elimina una carpeta y detecta el problema.

### Ejercicio 4 — Entorno
Muestra los comandos para crear y activar `.venv`.

### Ejercicio 5 — pyproject
Crea un `pyproject.toml` con dependencias runtime y dev.

### Ejercicio 6 — gitignore
Genera un `.gitignore` profesional.

### Ejercicio 7 — `.env.example`
Crea el fichero de plantilla.

### Ejercicio 8 — `.env`
Crea un `.env` local de demostración.

### Ejercicio 9 — python-dotenv
Carga el `.env` con `load_dotenv()`.

### Ejercicio 10 — Validación de entorno
Provoca una variable obligatoria ausente y detecta el error.

### Ejercicio 11 — JSON
Guarda y carga configuración JSON.

### Ejercicio 12 — YAML
Guarda y carga configuración YAML.

### Ejercicio 13 — Comparación
Explica cuándo usar `.env`, JSON o YAML.

### Ejercicio 14 — Config module
Crea `config.py` reusable.

### Ejercicio 15 — Pipeline module
Crea `pipeline.py`.

### Ejercicio 16 — Script
Crea `scripts/main.py`.

### Ejercicio 17 — Integridad
Detecta automáticamente ficheros obligatorios vacíos.

### Ejercicio 18 — Reto final
Construye un proyecto profesional completo que:

1. cree carpetas;
2. cree ficheros;
3. genere pyproject;
4. genere gitignore;
5. genere `.env.example`;
6. genere `.env`;
7. cargue `.env`;
8. valide variables obligatorias;
9. genere JSON;
10. genere YAML;
11. valide configuración;
12. cree paquete src;
13. cree scripts;
14. cree tests vacíos;
15. detecte inconsistencias;
16. las corrija;
17. imprima árbol;
18. documente instalación y ejecución.

## 24. Preguntas de reflexión

1. ¿Qué aporta la estructura `src/`?
2. ¿Qué diferencia hay entre entorno virtual y `pyproject.toml`?
3. ¿Por qué no versionar `.venv`?
4. ¿Qué diferencia hay entre runtime y dev dependencies?
5. ¿Por qué `pip freeze` no equivale a declarar dependencias?
6. ¿Qué debe contener `.env.example`?
7. ¿Por qué `.env` no debe versionarse?
8. ¿Qué hace `load_dotenv()`?
9. ¿Qué diferencia hay entre `.env`, JSON y YAML?
10. ¿Por qué no guardar secretos en JSON/YAML versionados?
11. ¿Qué ventaja tiene configuración externa?
12. ¿Qué aporta validar configuración?
13. ¿Por qué conviene un script de entrada?
14. ¿Qué ventaja tiene instalación editable?
15. ¿Qué hace que un proyecto sea reproducible?